# VideoMAE-Small — Diving48 V2 — **ver 8 (bản chạy trên tài khoản Google khác)** (pipeline giữ nguyên theo bản mẫu HMDB51)

## 👉 Hướng dẫn cho người chạy notebook (đọc trước khi Run all)

**1. Lấy dữ liệu về Drive của bạn.** Cần 4 file nằm chung một thư mục:
`Diving48_vocab.json`, `Diving48_V2_train.json`, `Diving48_V2_test.json`, `Diving48_rgb.tar.gz` (~9,6 GB).
- **Cách khuyên dùng:** chủ dữ liệu chia sẻ thư mục `video_classification_2026` cho email của bạn (quyền **Người xem** là đủ). Mở Google Drive → *Được chia sẻ với tôi* → chuột phải thư mục → **Sắp xếp → Thêm lối tắt** → chọn **Drive của tôi**. Lối tắt không tốn dung lượng Drive của bạn.
- Hoặc tự upload / tạo bản sao 4 file vào `Drive của tôi/video_classification_2026/` (tốn ~9,6 GB dung lượng của bạn).
- Đặt ở chỗ khác hoặc đổi tên thư mục cũng được: cell 3 sẽ **tự tìm** thư mục chứa đủ 4 file trong Drive của bạn (sâu tối đa 3 cấp); nếu không tìm thấy thì sửa `DRIVE_DATA_DIR` ở cell 3.

**2. Chọn GPU:** *Runtime → Change runtime type → GPU* (khuyên dùng **A100**; L4/T4 vẫn chạy được nhưng chậm hơn nhiều). Notebook dừng ngay ở cell 2 nếu chưa bật GPU.

**3. Runtime → Run all.** Khi được hỏi quyền Google Drive, chọn **đúng tài khoản có dữ liệu / lối tắt ở bước 1**.
- Lần đầu notebook giải nén `Diving48_rgb.tar.gz` thẳng từ Drive vào `/content` (khoảng 10–20 phút) rồi chia train/val/test 70/15/15 với `SEED = 42`. Cách chia là tất định (sắp xếp tên file rồi xáo với seed cố định) nên **ra đúng cùng một split** với các bản Base/Large ver 8 của chủ notebook.
- Mọi thứ cần giữ lại (cache dữ liệu, checkpoint từng epoch, kết quả) được ghi vào thư mục làm việc **trên Drive của bạn**: `DRIVE_WORK_DIR = MyDrive/Diving48_VideoMAE_Small_v8` (tự tạo). Thư mục dữ liệu ở bước 1 chỉ cần quyền đọc.
- Cache dữ liệu đã chia split (~10 GB) chỉ được lưu lên Drive nếu Drive của bạn còn đủ chỗ (`SAVE_DRIVE_CACHE = "auto"`). Không đủ chỗ thì bỏ qua, mỗi lần chạy lại sẽ giải nén lại từ tar.gz (không ảnh hưởng kết quả).
- Batch size tự chọn theo bộ nhớ GPU (≥35 GB: 8×6, ≥20 GB: 4×12, nhỏ hơn: 2×24; effective batch luôn = 48 như bản gốc) và được **ghi cố định** vào `batch_config.json` ở lần chạy đầu, nên resume trên GPU khác vẫn dùng đúng cấu hình cũ.

**4. Bị disconnect:** kết nối lại (GPU) rồi **Run all** lần nữa — notebook tự train tiếp từ checkpoint gần nhất trên Drive (chi tiết ở mục "Ver 7" phía dưới). **Không** sửa hyperparameter giữa các lần chạy.

**5. Gửi kết quả lại:** khi chạy xong, chia sẻ thư mục `MyDrive/Diving48_VideoMAE_Small_v8/final_outputs_diving48_v8_small` (CSV so sánh, logits test, model cuối) và các file `epoch_metrics.csv` trong `model_outputs_diving48_v8_small/<tên run>/` cho chủ notebook.

> Cảnh báo `HF_TOKEN` / "secret does not exist" khi tải model từ Hugging Face là bình thường (model công khai, không cần token).

**Khác biệt so với `Diving48_VideoMAE_Small_v8.ipynb` (bản chạy trên tài khoản của chủ notebook):** pipeline, split, hyperparameter, tên run giữ nguyên; chỉ sửa phần môi trường:
- Tách thư mục **dữ liệu nguồn** (`DRIVE_DATA_DIR`, chỉ đọc, tự dò tìm) và **thư mục làm việc** (`DRIVE_WORK_DIR`, có kiểm tra quyền ghi) cho checkpoint/kết quả/cache.
- Kiểm tra GPU ngay đầu notebook; kiểm tra dung lượng đĩa `/content` trước khi giải nén.
- Cache lên Drive có kiểm tra dung lượng còn trống (`SAVE_DRIVE_CACHE`), lưu dở thì xoá và train tiếp bình thường. Nếu chủ dữ liệu chia sẻ luôn thư mục cache `diving48_prepared_70_15_15` (nằm cạnh 4 file dữ liệu) thì notebook dùng thẳng, không cần giải nén.
- Batch size / `NUM_WORKERS` tự chọn theo GPU/CPU được cấp (giữ effective batch 48), khoá lại trong `batch_config.json` để resume nhất quán.

---

**Khác biệt so với ver 5:**
- **Sửa lỗi gradient accumulation**: `forward(..., **kwargs)` khiến `Trainer` (transformers 4.57) tưởng model tự chuẩn hoá loss theo `num_items_in_batch`, nên **không chia loss cho `gradient_accumulation_steps`** -> gradient bị nhân 12 lần so với trung bình và Training Loss hiển thị gấp 12 lần (ví dụ 42.9 thay vì ~3.6). Ver 6 đặt `accepts_loss_kwargs = False` cho cả 2 model để Trainer chia loss đúng. Nên chạy lại từ đầu, không dùng checkpoint của ver 5 trở về trước.

**Khác biệt so với ver 4:**
- **Cache trên Drive**: lần chạy đầu giải nén + chia split xong sẽ lưu bản đã chuẩn bị (`train/val/test` + `annotations.csv` + `label_map.json`) vào `DRIVE_DATA_DIR/diving48_prepared/`. Các lần chạy sau (và 4 notebook backbone còn lại) thấy cache là copy thẳng về `/content` bằng nhiều luồng, **không giải nén tar.gz và không chia split lại**. Muốn tạo lại từ đầu: xoá thư mục `diving48_prepared` trên Drive, hoặc đặt `USE_DRIVE_CACHE = False`.
- Tất cả backbone dùng chung đúng một bản split, nên kết quả so sánh công bằng hơn.

**Khác biệt so với ver 3:**
- Sửa lỗi `CUDA out of memory` trên A100 40GB: batch size nhỏ hơn + `GRADIENT_ACCUMULATION_STEPS` để effective batch vẫn = 48, `NUM_WORKERS=8` (Colab báo tối đa hợp lý là 12), bật `expandable_segments` giảm phân mảnh bộ nhớ GPU.

**Khác biệt so với ver 2:**
- Code tương thích cả `transformers` 4.x và 5.x: tự chọn `warmup_ratio` (4.x) hoặc `warmup_steps` dạng float (5.x), nên không còn lỗi `unexpected keyword argument 'warmup_ratio'`.
- Vẫn ghim `transformers<5` ở cell cài thư viện. Nếu runtime đang dùng transformers 5.x **trước 5.18** (nạp sai `q_bias`/`v_bias` của VideoMAE) thì dừng ngay và hướng dẫn Restart session; từ 5.18 trở lên thì cho chạy.

**Khác biệt so với ver 1:**
- Đường dẫn Drive mặc định: `MyDrive/video_classification_2026/`.
- Copy/giải nén dữ liệu bằng Python (`check=True`) nên **lỗi sẽ báo ngay**, không còn "lỗi im lặng" như lệnh `!cp`/`!tar`; giải nén thẳng từ Drive (không copy tar 9.6GB sang `/content`), và bỏ qua nếu đã giải nén rồi.
- Các hyperparameter khác (lr, epoch, dropout, weight decay, số clip...) giữ nguyên theo `nhap.txt`; batch size/worker đã chỉnh cho A100 40GB. Nếu GPU lớn hơn (A100 80GB) có thể tăng `TRAIN_BATCH_SIZE` và giảm `GRADIENT_ACCUMULATION_STEPS` tương ứng.

Notebook này giữ **nguyên vẹn pipeline** của bản mẫu VideoMAE-Large/HMDB51 (tổ chức dữ liệu theo thư mục lớp, `VideoFolderDataset`, 2 model `baseline` mean-pool + `proposed` event-level Transformer, cách train/test/report giống hệt). Phần **duy nhất** được viết thêm là bước chuyển đổi định dạng gốc của Diving48 (video phẳng + JSON annotation) sang cấu trúc thư mục theo lớp giống HMDB51, và cách xác định train/val/test dùng **đúng split chính thức của Diving48 V2** thay vì chia ngẫu nhiên theo tỉ lệ (vì HMDB51 không có split chuẩn nên bản mẫu tự chia, còn Diving48 đã có `Diving48_V2_train.json` / `Diving48_V2_test.json`).

Backbone của file này: **VideoMAE-Small** (`MCG-NJU/videomae-small-finetuned-kinetics`).

## Chuẩn bị dữ liệu trên Google Drive (dùng chung cho cả 5 backbone)

Upload vào `MyDrive/video_classification_2026/` (đổi `DRIVE_DATA_DIR` ở cell 3 nếu để chỗ khác):
- `Diving48_vocab.json`
- `Diving48_V2_train.json`
- `Diving48_V2_test.json`
- `Diving48_rgb.tar.gz` (video RGB đã cắt sẵn theo từng clip, tên file = `vid_name` trong annotation)


## Ver 8 — ⚠️ gộp train+test chính thức rồi chia lại 70/15/15 NGẪU NHIÊN THEO CLIP

> Bản này dùng để **so sánh với ver 9** (cùng pipeline, cùng hyperparameter, ver 9 chia theo nhóm video gốc). Diving48 chỉ có 84 video gốc, mỗi video cắt ra hàng trăm clip; chia ngẫu nhiên theo clip sẽ đưa clip của cùng một video vào cả train/val/test -> **rò rỉ dữ liệu, kết quả val/test có thể cao hơn thực tế**. Không dùng kết quả bản này làm kết quả chính.

Do độ dài video ở train và test chính thức của Diving48 V2 khác nhau, notebook này **gộp toàn bộ ~18.4k clip rồi chia lại ngẫu nhiên có phân tầng theo lớp thành train/val/test = 70/15/15** (`TRAIN_RATIO/VAL_RATIO/TEST_RATIO` ở cell cấu hình, `SEED = 42`). Lớp quá ít mẫu được xử lý riêng (xem `stratified_split_files`).

**Cần làm khi chạy lần đầu sau thay đổi này:**
- Cache dữ liệu đổi sang `diving48_prepared_70_15_15` nên cache split cũ (`diving48_prepared`) sẽ **không** bị dùng nhầm; lần đầu sẽ giải nén tar.gz và chia lại, rồi lưu cache mới.
- Thư mục checkpoint/kết quả đổi sang `model_outputs_diving48_v8_small` và `final_outputs_diving48_v8_small` (thêm hậu tố `_small` để không ghi đè kết quả/checkpoint của bản VideoMAE-Base / VideoMAE-Large ver 8 trên cùng Drive), vì checkpoint cũ được train trên split khác.
- **Kết quả không còn so sánh trực tiếp được với số liệu công bố** trên split chính thức của Diving48.

### Khác biệt của bản VideoMAE-Small so với VideoMAE-Base ver 8

- `BACKBONE_NAME = "MCG-NJU/videomae-small-finetuned-kinetics"` (hidden size 384, 12 layer, 6 head) — giữ đúng checkpoint của VideoMAE-Small ver 7. MCG-NJU không công bố checkpoint Small chỉ pre-train tự giám sát (như `videomae-base` / `videomae-large`), nên bản Small dùng checkpoint đã fine-tune trên Kinetics-400; `VideoMAEModel.from_pretrained` chỉ nạp phần encoder, bỏ head phân loại 400 lớp. Code model lấy `hidden_size` từ config nên không cần sửa; `event_encoder_heads=4` vẫn chia hết 384.
- `TRAIN_BATCH_SIZE = EVAL_BATCH_SIZE = 8`, `GRADIENT_ACCUMULATION_STEPS = 6` (effective batch vẫn = 48, giống VideoMAE-Small ver 7); số step mỗi epoch vẫn như bản Base. Nếu GPU còn dư bộ nhớ có thể tăng `TRAIN_BATCH_SIZE` và giảm `GRADIENT_ACCUMULATION_STEPS` tương ứng (giữ tích = 48).
- Các hyperparameter còn lại (lr 2e-5, dropout 0.25, weight decay 0.05, 30 epoch, patience 3, label smoothing 0.05, cùng split 70/15/15 `SEED = 42`) **giữ nguyên bản Base** để so sánh công bằng giữa các backbone.
- Tên run đổi thành `*_videomae_small_v8_*`; thư mục checkpoint/kết quả là `model_outputs_diving48_v8_small` / `final_outputs_diving48_v8_small`.
- **Tài nguyên:** mỗi checkpoint (kèm optimizer) chỉ khoảng vài trăm MB, train nhanh hơn bản Base đáng kể (Base proposed ~9,6 giờ GPU cho 30 epoch).

### Ver 7 (giữ nguyên trong ver 8) — thay đổi so với ver 6 và cách chạy tiếp khi bị disconnect

**Hyperparameter:** giữ nguyên ver 6 (batch 4 × accumulation 12 = 48, lr 2e-5, dropout 0.25, ...), chỉ đổi `NUM_EPOCHS` 16 -> **30** và early stopping `patience` 2 -> **3** (`EARLY_STOPPING_PATIENCE`).

**Lưu từng epoch lên Drive** (`MODEL_OUTPUT_ROOT = DRIVE_DATA_DIR/model_outputs_diving48_v7`):
- Mỗi epoch Trainer lưu `checkpoint-<step>` (model + optimizer + scheduler + RNG + trạng thái early stopping). Giữ checkpoint mới nhất + checkpoint tốt nhất (`save_total_limit=2`), mỗi checkpoint ~1 GB.
- `epoch_metrics.csv` trong thư mục của từng run: loss / F1 / Acc của mọi epoch.
- `TRAINING_DONE.json`: ghi khi run kết thúc (hết epoch hoặc early stopping dừng).
- Kết quả cuối (CSV so sánh, model đã lưu) ghi vào `FINAL_OUTPUT_DIR` trên Drive.

**Khi bị disconnect, làm đúng 3 bước:**
1. Kết nối lại runtime (cần GPU), chạy **Runtime > Run all** (hoặc chạy tuần tự từ cell đầu).
2. Cell mount Drive + copy cache dữ liệu về `/content` chạy lại như bình thường (vài phút, không giải nén tar.gz).
3. Cell train baseline/proposed **tự tìm checkpoint gần nhất trên Drive và train tiếp**. Run nào đã xong thì chỉ nạp lại best checkpoint (~3 phút evaluate) rồi chuyển sang run kế tiếp.

**Lưu ý:**
- **Không đổi** `NUM_EPOCHS`, batch size, accumulation, lr, ... giữa các lần resume (lịch LR và số step phụ thuộc vào chúng). Muốn train lại từ đầu: đổi tên `MODEL_OUTPUT_ROOT` hoặc xoá thư mục run tương ứng trên Drive.
- Mất tối đa phần tiến độ của 1 epoch đang dở (~30 phút) khi bị ngắt.
- Nếu resume báo lỗi do checkpoint mới nhất hỏng, xoá thư mục `checkpoint-<step>` mới nhất đó rồi chạy lại để lùi về epoch trước.
- Chạy tối đa 30 epoch × 2 model rất lâu trên Colab; early stopping có thể dừng sớm hơn.
- Checkpoint ver 6 (`/content/model_outputs_diving48`) không dùng được cho ver 7 (khác số epoch nên khác lịch LR).
- Cell 16/17 thêm `labels=` cho `classification_report` / `confusion_matrix` vì lớp `Forward_Dive_NoTwis_STR` không có mẫu nào (tránh `ValueError` ở `target_names`).


In [ ]:
# Kiểm tra GPU trước (không có GPU thì dừng ngay, khỏi mất thời gian giải nén dữ liệu)
import subprocess
_gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if _gpu.returncode != 0 or not _gpu.stdout.strip():
    raise RuntimeError("Runtime chưa có GPU. Vào Runtime > Change runtime type > chọn GPU (khuyên dùng A100), rồi Run all lại.")
print("GPU:", _gpu.stdout.strip())

from google.colab import drive
drive.mount('/content/drive')  # chọn đúng tài khoản Google có dữ liệu (hoặc lối tắt tới thư mục dữ liệu được chia sẻ)


In [ ]:
import os

MYDRIVE = "/content/drive/MyDrive"

# (1) Thư mục chứa 4 file dữ liệu gốc. Có thể là thư mục của bạn hoặc LỐI TẮT tới thư mục được chia sẻ (chỉ cần quyền xem).
#     Nếu không đúng, notebook tự dò trong Drive của bạn (sâu tối đa 3 cấp).
DRIVE_DATA_DIR = os.path.join(MYDRIVE, "video_classification_2026")  # <-- chỉnh lại nếu cần

# (2) Thư mục làm việc trên Drive CỦA BẠN: cache dữ liệu, checkpoint từng epoch, kết quả cuối (tự tạo nếu chưa có).
#     Giữ nguyên giữa các lần chạy để resume được.
DRIVE_WORK_DIR = os.path.join(MYDRIVE, "Diving48_VideoMAE_Small_v8")

# (3) Lưu cache dữ liệu đã chia split (~10 GB) lên Drive: True / False / "auto" (= chỉ lưu khi Drive còn đủ chỗ)
SAVE_DRIVE_CACHE = "auto"

SOURCE_FILES = [
    "Diving48_vocab.json",
    "Diving48_V2_train.json",
    "Diving48_V2_test.json",
    "Diving48_rgb.tar.gz",
]
CACHE_DIR_NAME = "diving48_prepared_70_15_15"  # cache split 70/15/15 (khác cache split chính thức cũ "diving48_prepared")
CACHE_ITEMS = ["train", "val", "test", "annotations.csv", "label_map.json"]
USE_DRIVE_CACHE = True


def _has_all_sources(d):
    return all(os.path.exists(os.path.join(d, f)) for f in SOURCE_FILES)


def _find_data_dir(roots, max_depth=3):
    """Tìm thư mục chứa đủ 4 file dữ liệu (lối tắt tới thư mục được chia sẻ cũng hiện như thư mục thường)."""
    for root in roots:
        if not os.path.isdir(root):
            continue
        base_depth = root.rstrip(os.sep).count(os.sep)
        for dirpath, dirnames, filenames in os.walk(root):
            if "Diving48_rgb.tar.gz" in filenames and _has_all_sources(dirpath):
                return dirpath
            if dirpath.count(os.sep) - base_depth >= max_depth:
                dirnames[:] = []
            else:
                # bỏ qua thư mục kết quả/cache (rất nhiều file nhỏ, duyệt chậm trên Drive)
                dirnames[:] = [d for d in dirnames if not d.startswith(("model_outputs", "final_outputs", "diving48_prepared", "."))]
    return None


# --- Thư mục làm việc: phải ghi được ---
try:
    os.makedirs(DRIVE_WORK_DIR, exist_ok=True)
    _probe = os.path.join(DRIVE_WORK_DIR, ".write_test")
    with open(_probe, "w") as f:
        f.write("ok")
    os.remove(_probe)
except OSError as e:
    raise PermissionError(
        f"Không ghi được vào {DRIVE_WORK_DIR} ({e}). DRIVE_WORK_DIR phải nằm trong Drive của chính bạn "
        "(không đặt trong thư mục người khác chia sẻ chỉ với quyền xem)."
    )
print("Thư mục làm việc (checkpoint / kết quả / cache):", DRIVE_WORK_DIR)

# --- Cache dữ liệu đã chia split: ưu tiên cache của bạn, sau đó cache nằm cạnh dữ liệu (nếu được chia sẻ kèm) ---
_cache_candidates = [os.path.join(DRIVE_WORK_DIR, CACHE_DIR_NAME)]
if os.path.isdir(DRIVE_DATA_DIR):
    _cache_candidates.append(os.path.join(DRIVE_DATA_DIR, CACHE_DIR_NAME))
DRIVE_CACHE_DIR = _cache_candidates[0]  # nơi sẽ lưu cache mới (luôn trên Drive của bạn)
CACHE_HIT = False
if USE_DRIVE_CACHE:
    for _c in _cache_candidates:
        if os.path.exists(os.path.join(_c, "READY")):
            DRIVE_CACHE_DIR, CACHE_HIT = _c, True
            break
CACHE_READY_FILE = os.path.join(DRIVE_CACHE_DIR, "READY")  # chỉ được ghi khi lưu cache xong hoàn toàn

if CACHE_HIT:
    print("Tìm thấy cache dữ liệu đã chuẩn bị sẵn:", DRIVE_CACHE_DIR, "-> bỏ qua giải nén và chia split.")
else:
    if not _has_all_sources(DRIVE_DATA_DIR):
        print(f"Không thấy đủ 4 file dữ liệu trong {DRIVE_DATA_DIR}, đang dò tìm trong Drive của bạn...")
        _found = _find_data_dir([MYDRIVE, "/content/drive/Shareddrives"])
        if _found is None:
            print("Các mục ở cấp đầu của MyDrive:")
            for f in sorted(os.listdir(MYDRIVE)):
                print("  *", f)
            raise FileNotFoundError(
                "Không tìm thấy thư mục chứa đủ 4 file " + ", ".join(SOURCE_FILES) + ". Kiểm tra lại: "
                "(1) đã mount đúng tài khoản Google chưa (Runtime > Disconnect and delete runtime rồi chạy lại để chọn lại tài khoản), "
                "(2) thư mục được chia sẻ đã được 'Thêm lối tắt' vào Drive của tôi chưa (mục 'Được chia sẻ với tôi' không được mount), "
                "(3) file .tar.gz 9,6 GB đã upload/đồng bộ xong chưa, "
                "(4) hoặc sửa DRIVE_DATA_DIR ở trên cho đúng đường dẫn."
            )
        DRIVE_DATA_DIR = _found
    print("Chưa có cache (hoặc USE_DRIVE_CACHE=False). Đã tìm thấy đủ file nguồn trong", DRIVE_DATA_DIR)
    for f in SOURCE_FILES:
        _p = os.path.join(DRIVE_DATA_DIR, f)
        print(f"  - {f}  ({os.path.getsize(_p) / 1024**3:.2f} GB)")


In [ ]:
import glob
import json
import shutil
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor

LOCAL_OUTPUT_ROOT = "/content/OUTPUT-diving48"  # phải trùng OUTPUT_ROOT ở cell cấu hình


def parallel_copy_tree(src_root, dst_root, workers=32, include_top=None):
    """Copy cây thư mục bằng nhiều luồng (đọc/ghi nhiều file nhỏ trên Drive rất chậm nếu copy tuần tự).
    Bỏ qua file đã có cùng kích thước nên chạy lại được sau khi bị ngắt. Trả về số file đã xử lý.
    include_top: chỉ lấy các mục cấp 1 nằm trong danh sách này (None = lấy hết). File READY luôn bị bỏ qua."""
    pairs = []
    for dirpath, _, files in os.walk(src_root):
        rel = os.path.relpath(dirpath, src_root)
        if include_top is not None and rel != "." and rel.split(os.sep)[0] not in include_top:
            continue
        out_dir = dst_root if rel == "." else os.path.join(dst_root, rel)
        os.makedirs(out_dir, exist_ok=True)
        for fname in files:
            if fname == "READY":
                continue
            if include_top is not None and rel == "." and fname not in include_top:
                continue
            pairs.append((os.path.join(dirpath, fname), os.path.join(out_dir, fname)))

    def _copy(pair):
        src, dst = pair
        if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
            return
        shutil.copyfile(src, dst)

    with ThreadPoolExecutor(max_workers=workers) as ex:
        list(ex.map(_copy, pairs))  # list() để lỗi copy (nếu có) được raise ngay
    return len(pairs)


if CACHE_HIT:
    # Dữ liệu đã giải nén + chia split sẵn trên Drive -> chỉ cần copy về /content
    with open(CACHE_READY_FILE, "r", encoding="utf-8") as f:
        _cache_info = json.load(f)
    t0 = time.time()
    print("Đang copy dữ liệu đã chuẩn bị từ Drive về /content ...")
    n_copied = parallel_copy_tree(DRIVE_CACHE_DIR, LOCAL_OUTPUT_ROOT, include_top=CACHE_ITEMS)
    print(f"Đã copy {n_copied} file trong {time.time() - t0:.0f}s")
    assert n_copied == _cache_info["n_files"], (
        f"Cache ghi nhận {_cache_info['n_files']} file nhưng chỉ copy được {n_copied}. "
        f"Xoá thư mục {DRIVE_CACHE_DIR} trên Drive (nếu là cache của bạn) hoặc đặt USE_DRIVE_CACHE = False để tạo lại."
    )
else:
    # Copy 3 file annotation (nhỏ) sang /content. Dùng Python nên thất bại sẽ raise lỗi ngay.
    for fname in ["Diving48_vocab.json", "Diving48_V2_train.json", "Diving48_V2_test.json"]:
        shutil.copy2(os.path.join(DRIVE_DATA_DIR, fname), os.path.join("/content", fname))

    # Giải nén video thẳng từ Drive (không copy tar.gz 9.6GB sang /content), bỏ qua nếu đã giải nén.
    VIDEO_TARGET_DIR = "/content/diving48_videos"
    EXPECTED_MIN_VIDEOS = 18000  # Diving48 có 18.404 clip

    def _count_videos(root):
        return len(glob.glob(os.path.join(root, "**", "*.mp4"), recursive=True))

    if _count_videos(VIDEO_TARGET_DIR) >= EXPECTED_MIN_VIDEOS:
        print("Video đã được giải nén trước đó, bỏ qua.")
    else:
        shutil.rmtree(VIDEO_TARGET_DIR, ignore_errors=True)
        os.makedirs(VIDEO_TARGET_DIR, exist_ok=True)
        # Giải nén (~10 GB) + chia split copy thêm 1 lần nữa (~10 GB) -> cần khoảng 25 GB trống trên đĩa của runtime.
        _free_gb = shutil.disk_usage("/content").free / 1024**3
        print(f"Dung lượng trống trên /content: {_free_gb:.0f} GB")
        if _free_gb < 25:
            raise RuntimeError(
                f"Đĩa của runtime chỉ còn {_free_gb:.0f} GB, cần khoảng 25 GB để giải nén và chia split. "
                "Đổi sang runtime GPU khác (A100/L4 có đĩa lớn hơn) hoặc Runtime > Disconnect and delete runtime rồi chạy lại."
            )
        print("Đang giải nén video từ Drive (vài phút)...")
        subprocess.run(
            ["tar", "-xzf", os.path.join(DRIVE_DATA_DIR, "Diving48_rgb.tar.gz"), "-C", VIDEO_TARGET_DIR],
            check=True,
        )

    n_videos = _count_videos(VIDEO_TARGET_DIR)
    print("Số video .mp4 trong", VIDEO_TARGET_DIR, ":", n_videos)
    assert n_videos >= EXPECTED_MIN_VIDEOS, f"Chỉ có {n_videos} video, mong đợi ~18.404 — giải nén có thể bị lỗi."


In [ ]:
# =========================
# 1) Cài thư viện
# =========================
# Ghim transformers<5: bản 5.x đã bỏ TrainingArguments(warmup_ratio) và (trước 5.18) nạp sai q_bias/v_bias của VideoMAE.
# Nếu runtime đã import transformers 5.x trước đó: Runtime > Restart session rồi Run all.
!pip -q install 'transformers<5' accelerate evaluate scikit-learn imageio[ffmpeg]


In [ ]:
!pip install av
!pip -q install av --no-binary av


In [ ]:
# =========================
# 2) Imports
# =========================
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')  # giảm phân mảnh bộ nhớ GPU

import av
import json
import random
import shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

from torch.utils.data import Dataset

from transformers import (
    Trainer,
    TrainingArguments,
    VideoMAEImageProcessor,
    VideoMAEModel,
)
from transformers.modeling_outputs import ImageClassifierOutput

from transformers import EarlyStoppingCallback

import transformers
from packaging import version

TF_VERSION = version.parse(transformers.__version__)
IS_TRANSFORMERS_V5 = TF_VERSION.major >= 5
print("transformers:", transformers.__version__)
if IS_TRANSFORMERS_V5 and TF_VERSION < version.parse("5.18.0"):
    raise RuntimeError(
        'Đang dùng transformers ' + transformers.__version__ + ' (5.x trước 5.18) nạp sai q_bias/v_bias của VideoMAE. '
        'Chạy Runtime > Restart session rồi Run all để cell cài đặt ghim transformers<5 có hiệu lực.'
    )

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# =========================
# 3) Cấu hình
# =========================

DIVING48_VOCAB_PATH = "/content/Diving48_vocab.json"
DIVING48_TRAIN_JSON_PATH = "/content/Diving48_V2_train.json"
DIVING48_TEST_JSON_PATH = "/content/Diving48_V2_test.json"
DIVING48_VIDEO_DIR = "/content/diving48_videos"

INPUT_ROOT = "/content/diving48"
OUTPUT_ROOT = "/content/OUTPUT-diving48"

# Ver 7: checkpoint từng epoch + kết quả cuối lưu THẲNG LÊN DRIVE để sống sót qua disconnect (/content bị xoá sạch).
# Đổi thư mục này = bắt đầu một lần train mới; giữ nguyên = tự động chạy tiếp từ checkpoint gần nhất.
# Ver 8: split dữ liệu đổi sang 70/15/15 -> dùng thư mục _v8 riêng, không resume nhầm checkpoint v7.
# Bản Small dùng hậu tố _small để không ghi đè checkpoint/kết quả của bản Base / Large ver 8.
# Bản chạy trên tài khoản khác: ghi vào DRIVE_WORK_DIR (Drive của người chạy), không ghi vào thư mục dữ liệu được chia sẻ.
MODEL_OUTPUT_ROOT = os.path.join(DRIVE_WORK_DIR, "model_outputs_diving48_v8_small")
FINAL_OUTPUT_DIR = os.path.join(DRIVE_WORK_DIR, "final_outputs_diving48_v8_small")

#######################################################################
FORCE_RESPLIT = False
SPLIT_ANNOTATION_FILE = "/content/OUTPUT-diving48/annotations.csv"

# Gộp train + test chính thức của Diving48 V2 rồi chia lại NGẪU NHIÊN CÓ PHÂN TẦNG theo lớp thành train/val/test = 70/15/15.
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9
SEED = 42

BACKBONE_NAME = "MCG-NJU/videomae-small-finetuned-kinetics"
NUM_FRAMES = 16
NUM_CLIPS = 4
NUM_EPOCHS = 30                 # ver 7: 16 -> 30 (KHÔNG đổi giữa chừng khi đang resume: lịch LR phụ thuộc số epoch)
EARLY_STOPPING_PATIENCE = 3     # ver 7: 2 -> 3
NUM_WORKERS = min(8, os.cpu_count() or 2)  # runtime ít CPU (T4/L4 thường 2–8 CPU) thì giảm theo
DROPOUT = 0.25
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.05
FREEZE_BACKBONE = False

# Effective batch = TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS = 48 (như bản mẫu: 24 * 2)
# Batch size tự chọn theo bộ nhớ GPU (A100 40GB: 8 x 6 như bản Small v8 gốc). Lần chạy đầu ghi vào batch_config.json,
# các lần resume sau dùng lại đúng giá trị đó dù Colab cấp GPU khác (đổi batch giữa chừng làm sai lịch train).
EFFECTIVE_BATCH_SIZE = 48
os.makedirs(MODEL_OUTPUT_ROOT, exist_ok=True)
BATCH_CONFIG_FILE = os.path.join(MODEL_OUTPUT_ROOT, "batch_config.json")
if os.path.exists(BATCH_CONFIG_FILE):
    with open(BATCH_CONFIG_FILE, "r", encoding="utf-8") as f:
        _batch_cfg = json.load(f)
    TRAIN_BATCH_SIZE = _batch_cfg["train_batch_size"]
    print("Dùng lại cấu hình batch đã khoá từ lần chạy trước:", _batch_cfg)
else:
    _gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    TRAIN_BATCH_SIZE = 8 if _gpu_mem_gb >= 35 else (4 if _gpu_mem_gb >= 20 else 2)
    with open(BATCH_CONFIG_FILE, "w", encoding="utf-8") as f:
        json.dump({"train_batch_size": TRAIN_BATCH_SIZE, "gpu": torch.cuda.get_device_name(0), "gpu_mem_gb": round(_gpu_mem_gb, 1)}, f)
    print(f"GPU {torch.cuda.get_device_name(0)} ({_gpu_mem_gb:.0f} GB) -> TRAIN_BATCH_SIZE = {TRAIN_BATCH_SIZE}")
EVAL_BATCH_SIZE = TRAIN_BATCH_SIZE
assert EFFECTIVE_BATCH_SIZE % TRAIN_BATCH_SIZE == 0
GRADIENT_ACCUMULATION_STEPS = EFFECTIVE_BATCH_SIZE // TRAIN_BATCH_SIZE
print(f"Batch {TRAIN_BATCH_SIZE} x accumulation {GRADIENT_ACCUMULATION_STEPS} = {EFFECTIVE_BATCH_SIZE}, NUM_WORKERS = {NUM_WORKERS}")

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".mpeg", ".mpg"}

os.makedirs(MODEL_OUTPUT_ROOT, exist_ok=True)
os.makedirs(FINAL_OUTPUT_DIR, exist_ok=True)


In [ ]:
# =========================
# 4) Utilities
# =========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def is_video_file(path: Path):
    return path.suffix.lower() in VIDEO_EXTS

def collect_videos_from_class(class_dir: Path):
    files = []
    if not class_dir.exists():
        return files
    for p in class_dir.rglob("*"):
        if p.is_file() and is_video_file(p):
            files.append(p)
    return sorted(files)

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def get_dynamic_classes(input_root: str):
    root = Path(input_root)
    classes = sorted([p.name for p in root.iterdir() if p.is_dir()])
    if not classes:
        raise ValueError(f"Không tìm thấy thư mục lớp nào trong {input_root}")
    return classes

def stratified_split_files(files, val_ratio, test_ratio, seed):
    """Chia danh sách video của MỘT lớp thành train/val/test theo tỉ lệ (chia phân tầng theo lớp).
    Xử lý an toàn khi lớp có rất ít mẫu (Diving48 có lớp chỉ vài mẫu, thậm chí không có mẫu nào):
      n = 0 -> rỗng; n = 1 -> train; n = 2 -> 1 train + 1 test;
      n >= 3 -> val và test mỗi tập tối thiểu 1 mẫu, train luôn còn tối thiểu 1 mẫu."""
    files = sorted(files)  # sort trước để kết quả không phụ thuộc thứ tự duyệt thư mục
    rng = random.Random(seed)
    rng.shuffle(files)

    n = len(files)
    if n == 0:
        return [], [], []
    if n == 1:
        return files, [], []
    if n == 2:
        return files[:1], [], files[1:]

    n_val = max(1, int(round(n * val_ratio)))
    n_test = max(1, int(round(n * test_ratio)))
    while n - n_val - n_test < 1:  # đảm bảo train còn ít nhất 1 mẫu
        if n_val >= n_test and n_val > 1:
            n_val -= 1
        else:
            n_test -= 1

    val_files = files[:n_val]
    test_files = files[n_val:n_val + n_test]
    train_files = files[n_val + n_test:]
    return train_files, val_files, test_files

def copy_files(file_list, dst_dir):
    ensure_dir(dst_dir)
    for src in file_list:
        dst = Path(dst_dir) / src.name
        if dst.exists():
            stem = dst.stem
            suffix = dst.suffix
            idx = 1
            while dst.exists():
                dst = Path(dst_dir) / f"{stem}_{idx}{suffix}"
                idx += 1
        shutil.copy2(src, dst)

def print_split_stats(stats_dict):
    print("=" * 80)
    print("THỐNG KÊ DATASET SPLIT")
    print("=" * 80)
    total_train = total_val = total_test = 0
    for cls, stats in stats_dict.items():
        print(
            f"{cls:40s} | train: {stats['train']:4d} | val: {stats['val']:4d} | test: {stats['test']:4d} | total: {stats['total']:4d}"
        )
        total_train += stats["train"]
        total_val += stats["val"]
        total_test += stats["test"]
    print("-" * 80)
    print(
        f"{'TOTAL':40s} | train: {total_train:4d} | val: {total_val:4d} | test: {total_test:4d} | total: {total_train + total_val + total_test:4d}"
    )
    print("=" * 80)

set_seed(SEED)


In [ ]:
# =========================
# 4.5) Chuyển đổi định dạng gốc của Diving48 (video phẳng + JSON annotation)
# sang cấu trúc thư mục theo lớp (INPUT_ROOT/<class_name>/<video>.mp4), giống HMDB51,
# để toàn bộ phần còn lại của pipeline (get_dynamic_classes, collect_videos_from_class,
# VideoFolderDataset, ...) dùng lại được nguyên vẹn, không phải viết lại.
# =========================
assert not (CACHE_HIT and FORCE_RESPLIT), f"Muốn chia split lại thì đặt USE_DRIVE_CACHE = False (hoặc xoá thư mục {DRIVE_CACHE_DIR} trên Drive)."
if CACHE_HIT:
    print('Dùng dữ liệu đã chuẩn bị từ cache trên Drive, bỏ qua bước chuyển đổi định dạng.')
else:
    _content_required = [DIVING48_VOCAB_PATH, DIVING48_TRAIN_JSON_PATH, DIVING48_TEST_JSON_PATH, DIVING48_VIDEO_DIR]
    _content_missing = [p for p in _content_required if not os.path.exists(p)]
    if _content_missing:
        print("Thiếu các đường dẫn sau trong /content:")
        for p in _content_missing:
            print("  -", p)
        print()
        print("Nội dung hiện có trong /content:")
        for f in sorted(os.listdir("/content")):
            print("  *", f)
        raise FileNotFoundError(
            "Thiếu dữ liệu trong /content. Nguyên nhân phổ biến nhất: Colab runtime đã bị "
            "disconnect/reset (ví dụ do idle quá lâu, hoặc bạn bấm Disconnect/Restart), nên toàn bộ "
            "/content bị xoá sạch, nhưng bạn chỉ chạy lại cell này mà chưa chạy lại các cell phía trên "
            "(mount Drive, copy file, giải nén video). Hãy dùng Runtime > Run all (hoặc chạy lại tuần tự "
            "từ cell đầu tiên) để tái tạo /content trước khi chạy cell này."
        )

    with open(DIVING48_VOCAB_PATH, "r", encoding="utf-8") as f:
        _vocab = json.load(f)
    DIVING48_CLASSES = ["_".join(row) for row in _vocab]  # tên lớp = ghép mô tả kỹ thuật lặn

    with open(DIVING48_TRAIN_JSON_PATH, "r", encoding="utf-8") as f:
        _official_train_entries = json.load(f)
    with open(DIVING48_TEST_JSON_PATH, "r", encoding="utf-8") as f:
        _official_test_entries = json.load(f)

    # Chỉ dùng 2 file JSON để lấy nhãn của toàn bộ video; việc chia train/val/test do cell split bên dưới quyết định.
    VIDNAME_TO_LABEL = {e["vid_name"]: e["label"] for e in (_official_train_entries + _official_test_entries)}

    # Idempotent: dùng os.path.lexists (không theo symlink) để không bị FileExistsError
    # khi symlink cũ bị "broken" (ví dụ chạy lại cell sau khi xoá/giải nén lại video gốc),
    # và luôn quét lại toàn bộ thay vì chỉ dựa vào "thư mục không rỗng" để quyết định bỏ qua
    # (lần chạy trước có thể đã dừng giữa chừng do lỗi, để lại dữ liệu chưa đầy đủ).
    print("Đang kiểm tra / tạo cấu trúc thư mục theo lớp cho Diving48...")
    for cls in DIVING48_CLASSES:
        ensure_dir(os.path.join(INPUT_ROOT, cls))

    _n_linked, _n_skipped_existing, _n_missing_label = 0, 0, 0
    for video_path in Path(DIVING48_VIDEO_DIR).rglob("*"):
        if not video_path.is_file() or not is_video_file(video_path):
            continue
        vid_name = video_path.stem
        label_id = VIDNAME_TO_LABEL.get(vid_name)
        if label_id is None:
            _n_missing_label += 1
            continue
        cls = DIVING48_CLASSES[label_id]
        link_path = Path(INPUT_ROOT) / cls / video_path.name

        if os.path.lexists(link_path):
            if not link_path.exists():
                # symlink cũ bị broken (video gốc không còn đúng vị trí) -> tạo lại
                link_path.unlink()
                os.symlink(video_path.resolve(), link_path)
                _n_linked += 1
            else:
                _n_skipped_existing += 1
            continue

        os.symlink(video_path.resolve(), link_path)
        _n_linked += 1

    print(f"Đã liên kết mới/sửa lại {_n_linked} video, bỏ qua {_n_skipped_existing} video đã có sẵn.")
    print(f"Tổng số lớp: {len(DIVING48_CLASSES)} tại {INPUT_ROOT}")
    if _n_linked + _n_skipped_existing == 0:
        raise RuntimeError(
            f"Không tìm thấy video nào trong {DIVING48_VIDEO_DIR}. Cell giải nén tar.gz phía trên có thể đã thất bại "
            "(lệnh ! không báo lỗi Python) — kiểm tra lại output của cell đó."
        )
    if _n_missing_label:
        print(f"Cảnh báo: {_n_missing_label} video không khớp annotation nào, đã bỏ qua.")


In [ ]:
# =========================
# 5) Tự động lấy danh sách lớp
# =========================
CLASSES = get_dynamic_classes(os.path.join(OUTPUT_ROOT, "train") if CACHE_HIT else INPUT_ROOT)
print("Detected classes:", len(CLASSES))
print("Number of classes:", len(CLASSES))


In [ ]:
# =========================
# 6) Tạo hoặc dùng lại train/val/test split
# Gộp toàn bộ video (train + test chính thức của Diving48 V2) rồi chia lại
# ngẫu nhiên có phân tầng theo lớp: train/val/test = 70/15/15 (seed cố định).
# =========================
input_root = Path(INPUT_ROOT)
output_root = Path(OUTPUT_ROOT)
SPLIT_ANNOTATION_FILE = output_root / "annotations.csv"

if SPLIT_ANNOTATION_FILE.exists() and not FORCE_RESPLIT:
    print(f"Đã tìm thấy split cũ: {SPLIT_ANNOTATION_FILE}")
    print("Sử dụng lại train/val/test split cũ, không chia lại dữ liệu.")

    df_ann = pd.read_csv(SPLIT_ANNOTATION_FILE)

else:
    print("Chưa có split cũ hoặc FORCE_RESPLIT=True. Tiến hành chia dữ liệu mới.")

    if output_root.exists():
        shutil.rmtree(output_root)

    stats = defaultdict(dict)

    for cls in CLASSES:
        class_dir = input_root / cls
        videos = collect_videos_from_class(class_dir)
        print(f"{cls}: found {len(videos)} videos")

        train_files, val_files, test_files = stratified_split_files(
            videos, VAL_RATIO, TEST_RATIO, SEED
        )

        copy_files(train_files, output_root / "train" / cls)
        copy_files(val_files, output_root / "val" / cls)
        copy_files(test_files, output_root / "test" / cls)

        stats[cls]["train"] = len(train_files)
        stats[cls]["val"] = len(val_files)
        stats[cls]["test"] = len(test_files)
        stats[cls]["total"] = len(videos)

    print("\nHoàn thành chia train/val/test.")
    print_split_stats(stats)


In [ ]:
# =========================
# 7) Tạo label map + annotations
# =========================
label2id = {cls: idx for idx, cls in enumerate(CLASSES)}
id2label = {idx: cls for cls, idx in label2id.items()}

mapping = {"label2id": label2id, "id2label": id2label}

with open(output_root / "label_map.json", "w", encoding="utf-8") as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)

rows = []
for split in ["train", "val", "test"]:
    for cls in CLASSES:
        class_dir = output_root / split / cls
        for video_path in collect_videos_from_class(class_dir):
            rows.append({
                "split": split,
                "video_path": str(video_path),
                "label": cls,
                "label_id": label2id[cls]
            })

if not SPLIT_ANNOTATION_FILE.exists() or FORCE_RESPLIT:
    df_ann = pd.DataFrame(rows)
    df_ann.to_csv(output_root / "annotations.csv", index=False, encoding="utf-8")
else:
    df_ann = pd.read_csv(output_root / "annotations.csv")

display(df_ann.head())
print(df_ann["split"].value_counts())


In [ ]:
# =========================
# 7.5) Lưu bản đã giải nén + chia split lên Drive để các lần chạy sau dùng lại ngay,
# không phải giải nén tar.gz và chia split nữa. Chỉ làm 1 lần.
# Bản chạy trên tài khoản khác: lưu vào DRIVE_WORK_DIR, chỉ khi Drive còn đủ chỗ (SAVE_DRIVE_CACHE = "auto");
# lưu thất bại (hết dung lượng...) thì xoá bản dở và vẫn train bình thường.
# =========================
def _tree_size_bytes(root, include_top):
    total = 0
    for item in include_top:
        p = os.path.join(root, item)
        if os.path.isfile(p):
            total += os.path.getsize(p)
        for dirpath, _, files in os.walk(p):
            total += sum(os.path.getsize(os.path.join(dirpath, f)) for f in files)
    return total


_do_save = USE_DRIVE_CACHE and not CACHE_HIT and SAVE_DRIVE_CACHE
if _do_save:
    _need_gb = _tree_size_bytes(OUTPUT_ROOT, CACHE_ITEMS) / 1024**3
    _reserve_gb = 5  # chừa chỗ cho checkpoint (2 run x 2 checkpoint) + kết quả cuối
    try:
        _drive_free_gb = shutil.disk_usage(DRIVE_WORK_DIR).free / 1024**3
    except OSError:
        _drive_free_gb = None
    print(f"Cache cần ~{_need_gb:.1f} GB; Drive còn trống: " + ("không rõ" if _drive_free_gb is None else f"{_drive_free_gb:.1f} GB"))
    if SAVE_DRIVE_CACHE == "auto" and _drive_free_gb is not None and _drive_free_gb < _need_gb + _reserve_gb:
        print(f"-> KHÔNG lưu cache: Drive không đủ chỗ (cần ~{_need_gb + _reserve_gb:.0f} GB tính cả checkpoint). "
              "Mỗi lần chạy lại sẽ giải nén lại từ tar.gz, kết quả không đổi.")
        _do_save = False

if _do_save:
    t0 = time.time()
    print(f"Đang lưu dữ liệu đã chuẩn bị lên {DRIVE_CACHE_DIR} (lần đầu mất một lúc, chỉ cần làm 1 lần)...")
    try:
        n_saved = parallel_copy_tree(OUTPUT_ROOT, DRIVE_CACHE_DIR, include_top=CACHE_ITEMS)
        with open(CACHE_READY_FILE, "w", encoding="utf-8") as f:
            json.dump({"n_files": n_saved}, f)
        print(f"Đã lưu {n_saved} file lên Drive trong {time.time() - t0:.0f}s. Lần sau notebook sẽ tự dùng cache này.")
    except OSError as e:
        print(f"Lưu cache thất bại ({e}) -> xoá bản lưu dở, tiếp tục train bình thường.")
        shutil.rmtree(DRIVE_CACHE_DIR, ignore_errors=True)
else:
    print("Bỏ qua lưu cache (đang dùng cache sẵn có, SAVE_DRIVE_CACHE/USE_DRIVE_CACHE tắt, hoặc Drive không đủ chỗ).")


In [ ]:
# =========================
# 8) Dataset + collator
# =========================
class VideoFolderDataset(Dataset):
    def __init__(
        self,
        root_dir,
        split,
        processor,
        label2id,
        num_frames=16,
        num_clips=4,
        train=True,
    ):
        self.root_dir = Path(root_dir)
        self.split = split
        self.processor = processor
        self.label2id = label2id
        self.num_frames = num_frames
        self.num_clips = num_clips
        self.train = train
        self.samples = self._gather_video_paths()

    def _gather_video_paths(self):
        samples = []
        for class_name, class_id in self.label2id.items():
            class_dir = self.root_dir / self.split / class_name
            if not class_dir.exists():
                continue
            for path in class_dir.rglob("*"):
                if path.is_file() and is_video_file(path):
                    samples.append((str(path), class_id))
        if not samples:
            raise ValueError(f"No videos found in {self.root_dir / self.split}")
        return sorted(samples)

    def __len__(self):
        return len(self.samples)

    def _read_video_frames(self, video_path):
      try:
          container = av.open(video_path)

          frames = []
          for frame in container.decode(video=0):
              frames.append(frame.to_ndarray(format="rgb24"))

          container.close()

          if len(frames) == 0:
              raise RuntimeError("Empty video")

          return np.stack(frames)

      except Exception as e:
          print(f"ERROR VIDEO: {video_path}")
          print(e)

          return np.zeros(
              (self.num_frames, 224, 224, 3),
              dtype=np.uint8
          )

    def _sample_clip_indices(self, total_frames):
        if total_frames < self.num_frames:
            base = np.linspace(0, total_frames - 1, self.num_frames)
            base = np.clip(np.round(base).astype(int), 0, total_frames - 1)
            return [base for _ in range(self.num_clips)]

        segment_boundaries = np.linspace(
            0, total_frames - self.num_frames, self.num_clips + 1
        ).astype(int)

        clips = []
        for i in range(self.num_clips):
            start_min = segment_boundaries[i]
            start_max = segment_boundaries[i + 1]
            if start_max < start_min:
                start_max = start_min

            if self.train:
                start = random.randint(start_min, start_max) if start_max > start_min else start_min
            else:
                start = (start_min + start_max) // 2

            indices = np.arange(start, start + self.num_frames)
            indices = np.clip(indices, 0, total_frames - 1)
            clips.append(indices)

        return clips

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        frames = self._read_video_frames(video_path)
        total_frames = frames.shape[0]
        clip_indices_list = self._sample_clip_indices(total_frames)

        clips_pixel_values = []
        for clip_indices in clip_indices_list:
            clip_frames = [frames[i] for i in clip_indices]
            encoded = self.processor(clip_frames, return_tensors="pt")
            pixel_values = encoded["pixel_values"].squeeze(0)
            clips_pixel_values.append(pixel_values)

        pixel_values = torch.stack(clips_pixel_values, dim=0)

        return {
            "pixel_values": pixel_values,
            "labels": torch.tensor(label, dtype=torch.long),
            "video_path": video_path,
        }

class VideoDataCollator:
    def __call__(self, features):
        pixel_values = torch.stack([f["pixel_values"] for f in features], dim=0)
        labels = torch.stack([f["labels"] for f in features], dim=0)
        video_paths = [f["video_path"] for f in features]
        return {"pixel_values": pixel_values, "labels": labels, "video_path": video_paths}


In [ ]:
# =========================
# 9) Models
# Improvements:
# - label smoothing is applied inside the model loss
# - Proposed model adds: residual + LayerNorm + clip positional embedding + 1-layer TransformerEncoder
# =========================
class MeanPoolVideoMAEClassifier(nn.Module):
    # Trainer chỉ chia loss cho gradient_accumulation_steps khi accepts_loss_kwargs=False.
    # (forward có **kwargs nên mặc định Trainer coi model tự chuẩn hoá loss -> gradient bị nhân GRADIENT_ACCUMULATION_STEPS lần)
    accepts_loss_kwargs = False

    def __init__(self, backbone_name, num_labels, dropout=0.25, freeze_backbone=False, label_smoothing=0.0):
        super().__init__()
        self.backbone = VideoMAEModel.from_pretrained(backbone_name)
        self.config = self.backbone.config
        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def _clip_encode(self, clip_pixel_values):
        outputs = self.backbone(pixel_values=clip_pixel_values)
        last_hidden_state = outputs.last_hidden_state
        clip_embedding = last_hidden_state.mean(dim=1)
        return clip_embedding

    def forward(self, pixel_values, labels=None, **kwargs):
        batch_size, num_clips = pixel_values.shape[:2]
        clip_embeddings = []

        for i in range(num_clips):
            clip_x = pixel_values[:, i]
            clip_emb = self._clip_encode(clip_x)
            clip_embeddings.append(clip_emb)

        clip_embeddings = torch.stack(clip_embeddings, dim=1)   # [B, C, D]
        video_repr = clip_embeddings.mean(dim=1)                # [B, D]
        logits = self.classifier(self.dropout(video_repr))

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return ImageClassifierOutput(loss=loss, logits=logits)


class EventAttentionPool(nn.Module):
    def __init__(self, hidden_size, dropout=0.25):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, x):
        # x: [B, C, D]
        weights = torch.softmax(self.score(x).squeeze(-1), dim=1)  # [B, C]
        pooled = torch.sum(x * weights.unsqueeze(-1), dim=1)       # [B, D]
        return pooled, weights


class EventLevelVideoUnderstandingModel(nn.Module):
    # Trainer chỉ chia loss cho gradient_accumulation_steps khi accepts_loss_kwargs=False.
    # (forward có **kwargs nên mặc định Trainer coi model tự chuẩn hoá loss -> gradient bị nhân GRADIENT_ACCUMULATION_STEPS lần)
    accepts_loss_kwargs = False

    def __init__(
        self,
        backbone_name,
        num_labels,
        dropout=0.25,
        freeze_backbone=False,
        label_smoothing=0.05,
        max_num_clips=8,
        event_encoder_layers=1,
        event_encoder_heads=4,
    ):
        super().__init__()
        self.backbone = VideoMAEModel.from_pretrained(backbone_name)
        self.config = self.backbone.config
        hidden_size = self.backbone.config.hidden_size

        self.event_projection = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size),
        )

        # Residual normalization after event projection
        self.event_norm = nn.LayerNorm(hidden_size)

        # Learnable temporal/clip positional embedding
        self.clip_pos_embed = nn.Parameter(torch.zeros(1, max_num_clips, hidden_size))
        nn.init.trunc_normal_(self.clip_pos_embed, std=0.02)

        # Event-level temporal reasoning over clip/event tokens
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=event_encoder_heads,
            dim_feedforward=hidden_size * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.event_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=event_encoder_layers,
        )

        self.event_pool = EventAttentionPool(hidden_size, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def _clip_encode(self, clip_pixel_values):
        outputs = self.backbone(pixel_values=clip_pixel_values)
        last_hidden_state = outputs.last_hidden_state
        clip_embedding = last_hidden_state.mean(dim=1)
        return clip_embedding

    def forward(self, pixel_values, labels=None, **kwargs):
        batch_size, num_clips = pixel_values.shape[:2]
        clip_embeddings = []

        for i in range(num_clips):
            clip_x = pixel_values[:, i]
            clip_emb = self._clip_encode(clip_x)
            clip_embeddings.append(clip_emb)

        clip_embeddings = torch.stack(clip_embeddings, dim=1)  # [B, C, D]

        # Event token construction: residual projection + normalization
        event_tokens = self.event_norm(
            clip_embeddings + self.event_projection(clip_embeddings)
        )

        # Add temporal order information
        if num_clips > self.clip_pos_embed.shape[1]:
            raise ValueError(
                f"num_clips={num_clips} exceeds max_num_clips={self.clip_pos_embed.shape[1]}. "
                "Increase max_num_clips in EventLevelVideoUnderstandingModel."
            )
        event_tokens = event_tokens + self.clip_pos_embed[:, :num_clips, :]

        # Event-level reasoning
        event_tokens = self.event_encoder(event_tokens)

        # Attention-based event aggregation
        video_repr, event_weights = self.event_pool(event_tokens)
        logits = self.classifier(self.dropout(video_repr))

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)

        output = ImageClassifierOutput(loss=loss, logits=logits)
        output.event_weights = event_weights
        output.event_tokens = event_tokens
        return output


In [ ]:
# =========================
# 10) Metrics: Top@1 và Top@5
# =========================
def topk_predictions(logits, k=5):
    """Return top-k predicted class indices for each sample."""
    k = min(k, logits.shape[1])
    return np.argsort(logits, axis=1)[:, -k:][:, ::-1]


def topk_accuracy(logits, labels, k=5):
    """Top-k accuracy: correct if the true label appears in top-k predictions."""
    topk = topk_predictions(logits, k)
    return np.mean([label in topk_i for label, topk_i in zip(labels, topk)])


def topk_macro_f1(logits, labels, k=5):
    """
    Top-k macro F1 for single-label multi-class classification.
    """
    num_classes = logits.shape[1]
    k = min(k, num_classes)
    topk = topk_predictions(logits, k)

    y_true = np.zeros((len(labels), num_classes), dtype=int)
    y_true[np.arange(len(labels)), labels] = 1

    y_pred = np.zeros((len(labels), num_classes), dtype=int)
    for i, pred_indices in enumerate(topk):
        y_pred[i, pred_indices] = 1

    return f1_score(y_true, y_pred, average="macro", zero_division=0)


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Top@1
    top1_preds = np.argmax(logits, axis=-1)
    top1_acc = accuracy_score(labels, top1_preds)
    top1_macro_f1 = f1_score(labels, top1_preds, average="macro", zero_division=0)

    # Top@5
    top5_acc = topk_accuracy(logits, labels, k=5)

    return {
        "top1_macro_f1": top1_macro_f1,
        "top1_acc": top1_acc,
        "top5_acc": top5_acc,
    }


In [ ]:
# =========================
# 11) Prepare processor + datasets
# =========================
processor = VideoMAEImageProcessor.from_pretrained(BACKBONE_NAME)

train_dataset = VideoFolderDataset(
    root_dir=OUTPUT_ROOT,
    split="train",
    processor=processor,
    label2id=label2id,
    num_frames=NUM_FRAMES,
    num_clips=NUM_CLIPS,
    train=True,
)

val_dataset = VideoFolderDataset(
    root_dir=OUTPUT_ROOT,
    split="val",
    processor=processor,
    label2id=label2id,
    num_frames=NUM_FRAMES,
    num_clips=NUM_CLIPS,
    train=False,
)

test_dataset = VideoFolderDataset(
    root_dir=OUTPUT_ROOT,
    split="test",
    processor=processor,
    label2id=label2id,
    num_frames=NUM_FRAMES,
    num_clips=NUM_CLIPS,
    train=False,
)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))


In [ ]:
# =========================
# 12) Hàm train chung (có lưu từng epoch lên Drive + tự chạy tiếp khi bị disconnect)
# Logic đúng cho publication:
# - train + validation trong train_model()
# - KHÔNG đánh giá test trong train_model()
# - test sẽ dùng trainer.predict(test_dataset) ở cell sau
#
# Cơ chế resume (ver 7):
# - Mỗi epoch Trainer lưu checkpoint (model + optimizer + scheduler + RNG + trainer_state + trạng thái early stopping)
#   vào MODEL_OUTPUT_ROOT/<run_name>/checkpoint-<step> trên Drive.
# - Chạy lại cell này (sau khi Run all các cell phía trên) -> tự tìm checkpoint hợp lệ gần nhất rồi train tiếp.
# - Khi 1 run kết thúc, file TRAINING_DONE.json được ghi -> lần sau bỏ qua train, chỉ nạp lại best checkpoint.
# - epoch_metrics.csv trong thư mục run ghi lại loss/F1/Acc của từng epoch (ghi lại toàn bộ lịch sử sau mỗi epoch).
# =========================
import re
from transformers import TrainerCallback

# transformers 5.x bỏ warmup_ratio: dùng warmup_steps dạng float (<1 = tỉ lệ), 4.x vẫn dùng warmup_ratio
WARMUP_KWARGS = {"warmup_steps": 0.1} if IS_TRANSFORMERS_V5 else {"warmup_ratio": 0.1}

def build_training_args(run_name, output_dir):
    return TrainingArguments(
        output_dir=output_dir,
        run_name=run_name,
        remove_unused_columns=False,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        **WARMUP_KWARGS,
        logging_steps=100,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        label_smoothing_factor=0.05,

        # compute_metrics trả về key "top1_macro_f1".
        # Không ghi "eval_" ở đây; Trainer sẽ tự thêm prefix "eval_" khi evaluate.
        metric_for_best_model="top1_macro_f1",
        greater_is_better=True,

        # Giữ checkpoint mới nhất + checkpoint tốt nhất (Trainer luôn bảo vệ best checkpoint khỏi bị xoá).
        save_total_limit=2,
        fp16=torch.cuda.is_available(),
        report_to="none",

        # An toàn cho cả NUM_WORKERS=0 và >0
        dataloader_num_workers=NUM_WORKERS,
        dataloader_pin_memory=True,
        dataloader_persistent_workers=(NUM_WORKERS > 0),
        dataloader_prefetch_factor=2 if NUM_WORKERS > 0 else None,
    )


class EpochProgressCallback(TrainerCallback):
    """Ghi kết quả từng epoch ra CSV trên Drive và đánh dấu run đã kết thúc (kể cả khi early stopping dừng sớm)."""

    def __init__(self, output_dir):
        self.output_dir = output_dir
        self.metrics_csv = os.path.join(output_dir, "epoch_metrics.csv")
        self.done_file = os.path.join(output_dir, "TRAINING_DONE.json")

    def _dump_log_history(self, state):
        os.makedirs(self.output_dir, exist_ok=True)
        # Ghi lại TOÀN BỘ log_history (được khôi phục khi resume) nên không bị trùng dòng sau khi chạy tiếp.
        pd.DataFrame(state.log_history).to_csv(self.metrics_csv, index=False)

    def on_evaluate(self, args, state, control, **kwargs):
        self._dump_log_history(state)

    def on_train_end(self, args, state, control, **kwargs):
        # Được gọi sau khi Trainer đã nạp lại best model (load_best_model_at_end).
        self._dump_log_history(state)
        with open(self.done_file, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "best_model_checkpoint": state.best_model_checkpoint,
                    "global_step": state.global_step,
                    "epoch": state.epoch,
                },
                f,
            )


def find_last_valid_checkpoint(output_dir):
    """Checkpoint-<step> mới nhất có đủ file để resume. Bỏ qua checkpoint ghi dở (ví dụ mất kết nối giữa chừng)."""
    if not os.path.isdir(output_dir):
        return None
    candidates = []
    for name in os.listdir(output_dir):
        m = re.fullmatch(r"checkpoint-(\d+)", name)
        path = os.path.join(output_dir, name)
        if m and os.path.isdir(path):
            candidates.append((int(m.group(1)), path))
    for _, path in sorted(candidates, reverse=True):
        has_weights = any(os.path.exists(os.path.join(path, w)) for w in ("model.safetensors", "pytorch_model.bin"))
        has_state = all(os.path.exists(os.path.join(path, f)) for f in ("optimizer.pt", "scheduler.pt", "trainer_state.json"))
        if has_weights and has_state:
            return path
        print("Bỏ qua checkpoint chưa đầy đủ:", path)
    return None


def load_weights_from_checkpoint(model, ckpt_dir):
    st_path = os.path.join(ckpt_dir, "model.safetensors")
    if os.path.exists(st_path):
        from safetensors.torch import load_file
        state_dict = load_file(st_path)
    else:
        state_dict = torch.load(os.path.join(ckpt_dir, "pytorch_model.bin"), map_location="cpu")
    model.load_state_dict(state_dict)


def train_model(model, run_name):
    output_dir = os.path.join(MODEL_OUTPUT_ROOT, run_name)
    args = build_training_args(run_name, output_dir)
    progress_cb = EpochProgressCallback(output_dir)

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=VideoDataCollator(),
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
            progress_cb,
        ],
    )

    if os.path.exists(progress_cb.done_file):
        # Run này đã train xong ở lần chạy trước -> chỉ nạp lại best checkpoint, không train lại.
        with open(progress_cb.done_file, "r", encoding="utf-8") as f:
            done_info = json.load(f)
        if not done_info.get("best_model_checkpoint"):
            raise RuntimeError(f"{progress_cb.done_file} không có best_model_checkpoint. Xoá file này để train lại/resume.")
        best_ckpt = os.path.join(output_dir, os.path.basename(done_info["best_model_checkpoint"]))
        print(f"[{run_name}] Đã train xong trước đó ({done_info['epoch']:.0f} epoch). Nạp best checkpoint: {best_ckpt}")
        load_weights_from_checkpoint(trainer.model, best_ckpt)
    else:
        resume_ckpt = find_last_valid_checkpoint(output_dir)
        if resume_ckpt:
            print(f"[{run_name}] Tìm thấy checkpoint, TIẾP TỤC train từ: {resume_ckpt}")
        else:
            print(f"[{run_name}] Chưa có checkpoint, train từ đầu.")
        trainer.train(resume_from_checkpoint=resume_ckpt)

    # Validation dùng evaluate để lấy metric chọn mô hình / log (model lúc này là best model).
    val_metrics = trainer.evaluate(eval_dataset=val_dataset)

    # Kiểm tra key metric để bảo đảm EarlyStopping hoạt động.
    print(f"{run_name} validation metric keys:", val_metrics.keys())

    return trainer, val_metrics


In [ ]:
# =========================
# 13) Train baseline
# (Nếu bị disconnect: Run all lại các cell phía trên rồi chạy cell này -> tự resume từ checkpoint trên Drive;
#  nếu baseline đã xong thì chỉ nạp lại best checkpoint rồi sang cell tiếp theo.)
# =========================
baseline_model = MeanPoolVideoMAEClassifier(
    backbone_name=BACKBONE_NAME,
    num_labels=len(CLASSES),
    dropout=DROPOUT,
    freeze_backbone=FREEZE_BACKBONE,
    label_smoothing=0.05,
)

baseline_trainer, baseline_val_metrics = train_model(
    baseline_model, "baseline_videomae_small_v8_meanpool_diving48"
)

print("Baseline val metrics:", baseline_val_metrics)


In [ ]:
# =========================
# 14) Train proposed model
# (Tự resume giống cell baseline.)
# =========================
proposed_model = EventLevelVideoUnderstandingModel(
    backbone_name=BACKBONE_NAME,
    num_labels=len(CLASSES),
    dropout=DROPOUT,
    freeze_backbone=FREEZE_BACKBONE,
    label_smoothing=0.05,
    max_num_clips=max(NUM_CLIPS, 8),
    event_encoder_layers=1,
    event_encoder_heads=4,
)

proposed_trainer, proposed_val_metrics = train_model(
    proposed_model, "proposed_videomae_small_v8_eventlevel_diving48"
)

print("Proposed val metrics:", proposed_val_metrics)


In [ ]:
# =========================
# 15) Test bằng predict(): một nguồn duy nhất cho bảng, report, confusion matrix
# =========================
def compute_test_result_from_predict(trainer, dataset, k=5):
    """
    Dùng predict() để lấy logits/labels, sau đó tự tính toàn bộ test metrics.
    Nhờ vậy bảng kết quả, classification_report và confusion matrix luôn khớp nhau.
    """
    trainer.model.eval()
    pred_output = trainer.predict(dataset)

    logits = pred_output.predictions
    labels = pred_output.label_ids

    top1_preds = np.argmax(logits, axis=-1)
    top1_acc = accuracy_score(labels, top1_preds)
    top1_macro_f1 = f1_score(labels, top1_preds, average="macro", zero_division=0)

    k = min(k, logits.shape[1])
    topk_preds = topk_predictions(logits, k=k)
    topk_acc = np.mean([label in topk_i for label, topk_i in zip(labels, topk_preds)])

    return {
        "pred_output": pred_output,
        "logits": logits,
        "labels": labels,
        "top1_preds": top1_preds,
        "topk_preds": topk_preds,
        "top1_acc": top1_acc,
        "top1_macro_f1": top1_macro_f1,
        "top5_acc": topk_acc,
    }


baseline_test_result = compute_test_result_from_predict(
    baseline_trainer, test_dataset, k=5
)

proposed_test_result = compute_test_result_from_predict(
    proposed_trainer, test_dataset, k=5
)

comparison_df = pd.DataFrame([
    {
        "Model": "Baseline",
        "Val Top@1 F1": baseline_val_metrics.get("eval_top1_macro_f1"),
        "Val Top@1 Acc": baseline_val_metrics.get("eval_top1_acc"),
        "Val Top@5 Acc": baseline_val_metrics.get("eval_top5_acc"),
        "Test Top@1 F1": baseline_test_result["top1_macro_f1"],
        "Test Top@1 Acc": baseline_test_result["top1_acc"],
        "Test Top@5 Acc": baseline_test_result["top5_acc"],
    },
    {
        "Model": "Proposed",
        "Val Top@1 F1": proposed_val_metrics.get("eval_top1_macro_f1"),
        "Val Top@1 Acc": proposed_val_metrics.get("eval_top1_acc"),
        "Val Top@5 Acc": proposed_val_metrics.get("eval_top5_acc"),
        "Test Top@1 F1": proposed_test_result["top1_macro_f1"],
        "Test Top@1 Acc": proposed_test_result["top1_acc"],
        "Test Top@5 Acc": proposed_test_result["top5_acc"],
    },
])

display(comparison_df)
comparison_csv_path = os.path.join(FINAL_OUTPUT_DIR, "model_comparison_top1_top5_consistent.csv")  # trên Drive
comparison_df.to_csv(comparison_csv_path, index=False)
print("Saved:", comparison_csv_path)


In [ ]:
# =========================
# 16) Classification report và Top@5 examples từ cùng kết quả predict()
# =========================
baseline_test_labels = baseline_test_result["labels"]
baseline_test_preds = baseline_test_result["top1_preds"]
baseline_test_top5 = baseline_test_result["topk_preds"]

proposed_test_labels = proposed_test_result["labels"]
proposed_test_preds = proposed_test_result["top1_preds"]
proposed_test_top5 = proposed_test_result["topk_preds"]

# Diving48 V2 có lớp không có mẫu nào (Forward_Dive_NoTwis_STR) -> sklearn chỉ thấy 47 nhãn trong khi CLASSES có 48 tên.
# Truyền labels=... để report/confusion matrix luôn đủ len(CLASSES) lớp, tránh ValueError ở target_names.
ALL_LABEL_IDS = np.arange(len(CLASSES))

print("Baseline classification report Top@1 (test)")
print(classification_report(
    baseline_test_labels,
    baseline_test_preds,
    labels=ALL_LABEL_IDS,
    target_names=CLASSES,
    zero_division=0
))

print("\nProposed classification report Top@1 (test)")
print(classification_report(
    proposed_test_labels,
    proposed_test_preds,
    labels=ALL_LABEL_IDS,
    target_names=CLASSES,
    zero_division=0
))


def make_top5_dataframe(labels, top5_preds, class_names, max_rows=20):
    rows = []
    for i, (y, pred_indices) in enumerate(zip(labels, top5_preds)):
        rows.append({
            "sample_id": i,
            "true_label": class_names[y],
            "top5_predictions": [class_names[j] for j in pred_indices],
            "hit_top5": bool(y in pred_indices),
        })
    return pd.DataFrame(rows).head(max_rows)

print("\nBaseline Top@5 examples:")
display(make_top5_dataframe(baseline_test_labels, baseline_test_top5, CLASSES))

print("\nProposed Top@5 examples:")
display(make_top5_dataframe(proposed_test_labels, proposed_test_top5, CLASSES))


In [ ]:
# =========================
# 17) Vẽ confusion matrices trên test
# =========================
def plot_confusion(cm, class_names, title):
    fig, ax = plt.subplots(figsize=(max(6, len(class_names) * 0.3), max(5, len(class_names) * 0.25)))
    ax.imshow(cm, interpolation="nearest")
    ax.set_title(title)
    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=90, fontsize=5)
    ax.set_yticklabels(class_names, fontsize=5)

    ax.set_ylabel("True label")
    ax.set_xlabel("Predicted label")
    plt.tight_layout()
    plt.show()

# labels=ALL_LABEL_IDS (định nghĩa ở cell 16) để ma trận luôn đủ len(CLASSES) hàng/cột, khớp với số tick.
baseline_cm = confusion_matrix(baseline_test_labels, baseline_test_preds, labels=ALL_LABEL_IDS)
proposed_cm = confusion_matrix(proposed_test_labels, proposed_test_preds, labels=ALL_LABEL_IDS)

plot_confusion(baseline_cm, CLASSES, "Baseline Confusion Matrix (Test)")
plot_confusion(proposed_cm, CLASSES, "Proposed Confusion Matrix (Test)")


In [ ]:
# =========================
# 18) Lưu model + processor (lên Drive)
# =========================
baseline_save_dir = os.path.join(FINAL_OUTPUT_DIR, "final_baseline_model_dynamic")
proposed_save_dir = os.path.join(FINAL_OUTPUT_DIR, "final_proposed_model_dynamic")

baseline_trainer.save_model(baseline_save_dir)
proposed_trainer.save_model(proposed_save_dir)
processor.save_pretrained(baseline_save_dir)
processor.save_pretrained(proposed_save_dir)

print("Saved baseline model to:", baseline_save_dir)
print("Saved proposed model to:", proposed_save_dir)
